# Chapter 7 — Exercise Solutions## RL for Recommendation Systems[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com)

### Solution 7.1: Slate-Based Recommendations

In [ ]:
import numpy as npfrom itertools import combinationsN_GENRES = 10GENRES = ['Action','Comedy','Drama','Sci-Fi','Horror','Romance',          'Thriller','Documentary','Animation','Fantasy']# Slate action space: all combinations of 3 genres from 10# C(10,3) = 120 slatesall_slates = list(combinations(range(N_GENRES), 3))print(f"Slate action space size: C(10,3) = {len(all_slates)}")print("Example slates:", [tuple(GENRES[g] for g in s) for s in all_slates[:3]])# For DQN: output layer has 120 units (one Q-value per slate)# Alternatively: use a scoring function f(genre_embeddings) + sum-poolingprint(f"\nDQN output layer: {len(all_slates)} units (one per slate)")print("Alternative: embed each genre, pool 3 embeddings, share parameters")print("  → Parameter-efficient but requires more careful architecture design")# Reward for a slate: max engagement across the 3 recommended genresclass SlateUserEnvironment:    def __init__(self):        self.preferences = np.random.dirichlet(np.ones(N_GENRES) * 2)    def step(self, slate):        """Slate = tuple of 3 genre indices. Reward = best engagement in slate."""        rewards = []        for g in slate:            p_watch = self.preferences[g]            watched = np.random.random() < p_watch            rewards.append(3.0 if (watched and np.random.random() < 0.6) else                           1.0 if watched else -0.5)        return max(rewards)  # user picks the best from the slateenv = SlateUserEnvironment()test_slate = (0, 2, 7)   # Action, Drama, Documentaryr = env.step(test_slate)print(f"\nTest slate {[GENRES[g] for g in test_slate]}: reward = {r}")# YOUR TURN: Implement a DQN with 120-unit output for slate selection# Measure: does slate DQN outperform single-genre DQN by > 15% reward?

### Solution 7.2: Short-term vs Long-term Gamma

In [ ]:
import numpy as np, matplotlib.pyplot as pltimport torch, torch.nn as nn, randomfrom collections import dequeclass UserEnv:    def __init__(self):        self.pref = np.random.dirichlet(np.ones(10)*2)        self.history=[]; self.step_count=0    def reset(self):        self.pref=np.random.dirichlet(np.ones(10)*2); self.history=[]; self.step_count=0        return np.zeros(13)    def step(self, genre):        p = self.pref[genre]        watched = np.random.random() < p        r = (3.0 if np.random.random()<0.5 else 1.0) if watched else -0.5        self.history.append(genre); self.step_count+=1        done = self.step_count >= 20        state = np.zeros(13)        state[:10] = self.pref + np.random.normal(0,0.05,10)        for i,g in enumerate(self.history[-3:]): state[10+i]=g/10        return state, r, doneclass RecNet(nn.Module):    def __init__(self):        super().__init__()        self.net = nn.Sequential(nn.Linear(13,64),nn.ReLU(),nn.Linear(64,64),nn.ReLU(),nn.Linear(64,10))    def forward(self,x): return self.net(x)def train_rec_dqn(gamma, n_episodes=300):    agent = RecNet(); target = RecNet()    target.load_state_dict(agent.state_dict())    opt = torch.optim.Adam(agent.parameters(), lr=1e-3)    buf = deque(maxlen=3000); eps=0.5; rewards=[]    for ep in range(n_episodes):        env=UserEnv(); s=env.reset(); ep_r=0        while True:            if random.random()<eps: a=random.randint(0,9)            else:                with torch.no_grad(): a=agent(torch.FloatTensor(s)).argmax().item()            ns,r,done=env.step(a); buf.append((s,a,r,ns,done)); ep_r+=r; s=ns            if len(buf)>=64:                batch=random.sample(buf,64)                sb,ab,rb,nb_,db=zip(*batch)                st=torch.FloatTensor(sb); at=torch.LongTensor(ab)                rt=torch.FloatTensor(rb); nt=torch.FloatTensor(nb_); dt=torch.FloatTensor(db)                q=agent(st).gather(1,at.unsqueeze(1)).squeeze()                with torch.no_grad(): nq=target(nt).max(1)[0]                tgt=rt+gamma*nq*(1-dt)                loss=nn.functional.mse_loss(q,tgt)                opt.zero_grad(); loss.backward(); opt.step()            if done: break        rewards.append(ep_r); eps=max(0.05,eps*0.995)        if ep%50==0: target.load_state_dict(agent.state_dict())    return rewardsprint("Training γ=0.5 (shortsighted)...")r_low  = train_rec_dqn(gamma=0.5)print("Training γ=0.99 (far-sighted)...")r_high = train_rec_dqn(gamma=0.99)w=20fig,ax=plt.subplots(figsize=(10,4))ax.plot(np.convolve(r_low, np.ones(w)/w,'valid'),  label='γ=0.5 (short-term)', color='tomato',   lw=2)ax.plot(np.convolve(r_high,np.ones(w)/w,'valid'), label='γ=0.99 (long-term)', color='seagreen', lw=2)ax.set_xlabel('Episode'); ax.set_ylabel('Smoothed Session Reward')ax.set_title('Short-term vs Long-term Reward Optimisation'); ax.legend(); ax.grid(True,alpha=0.3)plt.tight_layout(); plt.show()print(f"\nγ=0.5  final mean: {np.mean(r_low[-50:]):.2f}")print(f"γ=0.99 final mean: {np.mean(r_high[-50:]):.2f}")print("Expected: γ=0.99 learns to build engagement across the session")print("          γ=0.5  optimises immediate next click, ignores long-run engagement")

### Solution 7.3: Thompson Sampling Recommender

In [ ]:
import numpy as np, matplotlib.pyplot as pltN_GENRES = 10; N_STEPS = 2000class ThompsonSamplingRecommender:    """Beta-Bernoulli Thompson Sampling for binary watch/skip feedback."""    def __init__(self, n_arms):        self.alpha = np.ones(n_arms)  # successes + 1        self.beta  = np.ones(n_arms)  # failures + 1    def recommend(self):        samples = np.random.beta(self.alpha, self.beta)        return np.argmax(samples)    def update(self, genre, watched):        if watched: self.alpha[genre] += 1        else:       self.beta[genre]  += 1    @property    def estimated_probs(self):        return self.alpha / (self.alpha + self.beta)# Simulate user with fixed preferencesnp.random.seed(42)true_prefs = np.random.dirichlet(np.ones(N_GENRES)*2)best_genre = np.argmax(true_prefs)ts = ThompsonSamplingRecommender(N_GENRES)cumulative_reward = 0rewards_ts, rewards_random = [], []for step in range(N_STEPS):    # Thompson Sampling    genre_ts = ts.recommend()    watched_ts = np.random.random() < true_prefs[genre_ts]    ts.update(genre_ts, watched_ts)    r_ts = 1.0 if watched_ts else -0.5    cumulative_reward += r_ts    rewards_ts.append(cumulative_reward)    # Random baseline    genre_rand = np.random.randint(N_GENRES)    watched_rand = np.random.random() < true_prefs[genre_rand]    rewards_random.append((step+1) * (true_prefs.mean() - 0.5))print(f"Best genre: {genre} (true p={true_prefs[best_genre]:.3f})")print(f"TS estimated: {ts.estimated_probs[best_genre]:.3f}")print(f"TS cumulative reward: {cumulative_reward:.1f}")print(f"TS vs DQN: TS wins on simple stationary preferences")print(f"           DQN wins when preferences shift over the session")# YOUR TURN: Compare TS vs DQN when user preferences SHIFT at step 1000# (e.g. multiply preferences by a rotation matrix)# Does TS adapt? Does DQN adapt?